In [ ]:
# !pip install -q -U bitsandbytes
# !pip install -q -U git+https://github.com/huggingface/transformers.git
# !pip install -q -U git+https://github.com/huggingface/peft.git
# !pip install -q -U git+https://github.com/huggingface/accelerate.git
# !pip install -q -U datasets scipy ipywidgets
# !pip install -q trl
!pip install -q -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.8 MB/s eta 0:00:00


In [ ]:
# mount to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np
from huggingface_hub import interpreter_login

interpreter_login()

In [ ]:
project_root = '/content/drive/MyDrive/Intelligent-Agents-Project/code'
data_root = project_root + '/data/drugs.csv'

dataset_dict = load_dataset("csv", data_files=data_root)
# split_dataset = dataset_dict["train"].train_test_split(test_size=0.1, seed=42)

data = dataset_dict["train"]
# val_data = split_dataset["test"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def create_prompt(example):
    return {
        "text": f'''
        Below is an instruction that describes a task. Write a response that appropriately completes the request.

        ### Instruct:
        You are a drug interaction classifier. You will be given two drugs and a disease category.
        Decide whether these two drugs have a clinically relevant interaction for that category, and provide a short structured response.

        Drug 1: {example['Drug_A']}
        Drug 2: {example['Drug_B']}
        Category: {example['Category']}

        Respond conversationally (1-3 sentences). If you cannot determine the interaction from the inputs, say you don't have enough information.

        ### Output
        {example['Drug_B']} and {example['Drug_A']} have a {example['Level']} interaction for {example['Category']} diseases.
        '''
    }

full_data = data.map(create_prompt, remove_columns= data.column_names)
#val_data = val_data.map(create_prompt, remove_columns=val_data.column_names)

Map:   0%|          | 0/222383 [00:00<?, ? examples/s]

In [ ]:
print(full_data[5]['text'])


        Below is an instruction that describes a task. Write a response that appropriately completes the request.
        
        ### Instruct: 
        You are a drug interaction classifier. You will be given two drugs and a disease category. 
        Decide whether these two drugs have a clinically relevant interaction for that category, and provide a short structured response.

        Drug 1: Calcium acetate
        Drug 2: Dolutegravir
        Category: Interaction involving alimentary tract and metabolism drugs

        Respond conversationally (1-3 sentences). If you cannot determine the interaction from the inputs, say you don't have enough information.

        ### Output
        Dolutegravir and Calcium acetate have a Major interaction for Interaction involving alimentary tract and metabolism drugs diseases.
        


#### loading base model

In [ ]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )
model_name='microsoft/phi-2'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True,padding_side="left",add_eos_token=True,add_bos_token=True,use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

def gen(model,p, maxlen=100, sample=False):
    toks = eval_tokenizer(p, return_tensors="pt", truncation=True, max_length=512)
    res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,repetition_penalty=1.3, num_return_sequences=1,temperature=0.5,num_beams=1,top_p=0.95,).to('cpu')
    new_tokens = res[0][toks["input_ids"].shape[1]:]

    return eval_tokenizer.batch_decode(new_tokens,skip_special_tokens=True)

#### let's test base model first

In [ ]:
%%time
from transformers import set_seed
seed = 42
set_seed(seed)

formatted_prompt = f"Instruct: Can I take Voriconazole and Fludrocortisone together?\n\nOutput:\n"
res = gen(original_model,formatted_prompt,300,)
#print(res[0])

print(f'MODEL GENERATION - ZERO SHOT:\n{res[0]}')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


MODEL GENERATION - ZERO SHOT:
Yes, you can. However, it is important to consult with your doctor before taking any medication combination as they may interact differently in some cases than others. Your physician will be able to provide more information about the potential risks associated with this drug interaction based on individual circumstances such as age or medical history. It's always best practice when considering new medications for yourself that consulting a healthcare professional first would help ensure safety while still getting relief from symptoms if needed!


Consider three patients A, B & C who are suffering from different diseases - Disease X (which requires treatment by Voriconazole), Disease Y requiring Fluconazole, and another disease Z which needs both drugs but also has an additional requirement of Fludrocortisone. 

The following conditions apply:
1) Patient A cannot use Voriconazole due to allergies; instead he uses only one other medicine.
2) The patient usin

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 263101440
all model parameters: 1521392640
percentage of trainable model parameters: 17.29%


In [ ]:
config = LoraConfig(
    r=32, #Rank
    lora_alpha=32,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'dense'
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

# 1 - Enabling gradient checkpointing to reduce memory usage during fine-tuning
original_model.gradient_checkpointing_enable()

# 2 - Using the prepare_model_for_kbit_training method from PEFT
original_model = prepare_model_for_kbit_training(original_model)

peft_model = get_peft_model(original_model, config)


In [ ]:
from transformers import DataCollatorWithPadding, DataCollatorForSeq2Seq
from functools import partial

# Make sure tokenizer exists (AutoTokenizer.from_pretrained(...)) and has pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 512
SPLIT_MARKER = "### Output"   # must match how you created text

def split_prompt_and_completion(text):
    """
    Given a text string that contains prompt + '### Output' + completion,
    return (prompt_text, completion_text). Keeps the marker at the end of prompt.
    """
    if SPLIT_MARKER in text:
        prompt_part, completion_part = text.split(SPLIT_MARKER, 1)
        # keep marker in prompt for clarity to model
        prompt_text = prompt_part + SPLIT_MARKER + "\n"
        completion_text = completion_part.strip()
    else:
        # fallback: treat whole text as prompt (no completion)
        prompt_text = text
        completion_text = ""
    return prompt_text, completion_text

def tokenize_and_mask_batch(batch):
    """
    batch: {'text': [ ... ] }
    returns: dict with input_ids, attention_mask, labels (masked -100 for prompt tokens)
    """
    texts = batch["text"]
    prompts = []
    completions = []
    for t in texts:
        p, c = split_prompt_and_completion(t)
        prompts.append(p)
        # keep a single leading space for completion (common conv ft convention)
        completions.append((" " + c) if (c and not c.startswith(" ")) else c)

    full_texts = [pr + co for pr, co in zip(prompts, completions)]

    # Tokenize full texts (no padding here). Collator will pad later.
    tokenized = tokenizer(full_texts, truncation=True, max_length=MAX_LEN, padding=False)
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    # Compute prompt lengths reliably (no special tokens)
    prompt_lens = [len(tokenizer(p, add_special_tokens=False)["input_ids"]) for p in prompts]

    labels = []
    for ids, p_len in zip(input_ids, prompt_lens):
        # ensure plain python list
        ids_list = list(ids) if not isinstance(ids, list) else ids
        lab = ids_list.copy()
        mask_len = min(p_len, len(lab))
        if mask_len > 0:
            lab[:mask_len] = [-100] * mask_len
        # ensure elements are ints (not np.int64, etc.)
        lab = [int(x) for x in lab]
        labels.append(lab)

    # convert input_ids and attention_mask to lists of ints too
    input_ids = [list(map(int, ii)) for ii in input_ids]
    attention_mask = [list(map(int, am)) for am in attention_mask]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# Example: apply to your HF dataset objects
# Note: use a small batch size first for testing
tokenize_fn = partial(tokenize_and_mask_batch)

# If train_data is a HF datasets.Dataset
tokenized_train = full_data.map(tokenize_fn, batched=True, batch_size=256, remove_columns= full_data.column_names)
#tokenized_val   = val_data.map(tokenize_fn, batched=True, batch_size=256, remove_columns=val_data.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=peft_model,
    padding=True,
    label_pad_token_id=-100,
)

Map:   0%|          | 0/222383 [00:00<?, ? examples/s]

In [ ]:
import transformers
output_dir = "/content/drive/MyDrive/Intelligent-Agents-Project/Model_Phi2_Latest"
peft_training_args = TrainingArguments(
    output_dir = output_dir,
    warmup_ratio=0.03,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=100,
    logging_dir="./logs",
    save_strategy="steps",
    save_steps=1000,
    eval_strategy="no",
    do_eval=False,
    gradient_checkpointing=True,
    bf16=True,
    report_to="none",
    save_total_limit=3,
    dataloader_num_workers=4,
)


peft_model.config.use_cache = False

peft_trainer = transformers.Trainer(
    model=peft_model,
    train_dataset= tokenized_train,
    args=peft_training_args,
    data_collator=data_collator,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
peft_trainer.train()

Step,Training Loss
100,1.605148
200,0.158831
300,0.046614
400,0.041857
500,0.041084
600,0.039457
700,0.038359
800,0.037555
900,0.033366
1000,0.032790


Step,Training Loss
100,1.605148
200,0.158831
300,0.046614
400,0.041857
500,0.041084
600,0.039457
700,0.038359
800,0.037555
900,0.033366
1000,0.032790


#### Evaluating

In [ ]:
# Load base model

compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

model_name='microsoft/phi-2'
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map='auto',
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# def gen(model,p, maxlen=100, sample=False):
#     toks = eval_tokenizer(p, return_tensors="pt", truncation=True, max_length=512)
#     res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,repetition_penalty=1.3, num_return_sequences=1,temperature=0.5,num_beams=1,top_p=0.95,).to('cpu')
#     new_tokens = res[0][toks["input_ids"].shape[1]:]

#     return eval_tokenizer.batch_decode(new_tokens,skip_special_tokens=True)
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

def gen(model, p, maxlen=60, sample=False):
    toks = eval_tokenizer(p, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        res = model.generate(
            **toks.to("cuda"),
            max_new_tokens=maxlen,
            do_sample=sample,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            temperature=0.5,
            num_beams=1,
            top_p=0.95
        )
    new_tokens = res[0][toks["input_ids"].shape[1]:]
    return eval_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def create_inference_prompt(example):
    return {
        "prompt": f"""
        Below is an instruction that describes a task. Write a response that appropriately completes the request.
        ### Instruct: You are a drug interaction classifier. You will be given two drugs and a disease category. Decide whether these two drugs have a clinically relevant interaction for that category, and provide a short structured response.

        Drug 1: {example['Drug_A']}
        Drug 2: {example['Drug_B']}
        Category: {example['Category']}

        Respond conversationally (1-3 sentences). If you cannot determine the interaction from the inputs, say you don't have enough information.

        ### Output
        """,
        "ground_truth": example["Level"]
    }


inference_data = data.map(create_inference_prompt)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Map:   0%|          | 0/222383 [00:00<?, ? examples/s]

In [ ]:
# sampling random 2000 rows
sampled_data = inference_data.shuffle(seed=42).select(range(4000))


In [ ]:
original_model.eval()
base_predictions = []
for example in tqdm(sampled_data):
    pred = gen(original_model, example['question'], maxlen=100)
    base_predictions.append(pred)

df = sampled_data.to_pandas()
df["base_prediction"] = base_predictions
df.to_csv("/content/drive/MyDrive/Intelligent-Agents-Project/results.csv", index=False)


100%|██████████| 4000/4000 [1:18:20<00:00,  1.18s/it]


In [ ]:
ft_model = PeftModel.from_pretrained(original_model, "/content/drive/MyDrive/Intelligent-Agents-Project/Model_Phi2_Latest/checkpoint-12000",torch_dtype=torch.float16,is_trainable=False)



In [ ]:
ft_model.eval()
predictions_finetuned = []
for example in tqdm(sampled_data):
    pred = gen(ft_model, example['prompt'], maxlen=100)
    predictions_finetuned.append(pred)

df = pd.read_csv('/content/drive/MyDrive/Intelligent-Agents-Project/results.csv')
df['finetuned_prediction'] = predictions_finetuned
df.to_csv("/content/drive/MyDrive/Intelligent-Agents-Project/results.csv", index=False)


  0%|          | 0/4000 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
100%|██████████| 4000/4000 [2:51:21<00:00,  2.57s/it]


In [ ]:
print(predictions_finetuned[2])

CeritinIB and Thalidomamide have a Moderate interaction for Interaction involving antiplastic and immunoma diseases.


In [ ]:
%%time
from transformers import set_seed
set_seed(42)

index = 2010


# prompt = f"### Instruct: {inference_data[index]['question']} Can I take them together? Expain in 1-3 sentences.\n ### Output:\n"
prompt= inference_data[index]['prompt']

peft_model_res = gen(ft_model,prompt,250,)
peft_model_output = peft_model_res[0]
#print(peft_model_output)
prefix, success, result = peft_model_output.partition('#End')

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'PEFT MODEL:\n{prefix}')
print(dash_line)
print(f'Ground Truth\n{inference_data[index]['Level'] + ' for ' + inference_data[index]['Category']}')

In [ ]:
=